# Solutions - Deep RL with Stable-Baselines3

Solutions to the two exercises in `sb3_quickstart.ipynb`. Run the install cell of the main notebook first if you haven't.

## Exercise 1 - MountainCar with the WRONG (CartPole) hyperparameters

Expected result: evaluation stays at **-200.0 +/- 0.0** - the car never reaches the flag. The reward is -1 per step with nothing to guide the agent until the first success, and CartPole's settings explore too little for that success to happen within 50k steps.

In [1]:
import gymnasium as gym
from stable_baselines3 import DQN
from stable_baselines3.common.evaluation import evaluate_policy

env = gym.make('MountainCar-v0', render_mode='rgb_array')

cartpole_settings = dict(
    learning_rate=0.0023, gamma=0.99, batch_size=64,
    exploration_fraction=0.16, exploration_final_eps=0.04,
    buffer_size=100_000, learning_starts=1000, target_update_interval=10,
    train_freq=256, gradient_steps=128, policy_kwargs={'net_arch': [256, 256]},
)

model = DQN('MlpPolicy', env, seed=42, verbose=0, **cartpole_settings)
model.learn(total_timesteps=50_000, progress_bar=True)
mean, std = evaluate_policy(model, env, n_eval_episodes=20, deterministic=True)
print(f"CartPole settings on MountainCar: {mean:.1f} +/- {std:.1f}   (expected: stuck at -200)")

/Users/david.goll/Documents/projects/workshop-rl1-introduction/examples/.venv/lib/python3.13/site-packages/rich/liv
e.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/Users/david.goll/Documents/projects/workshop-rl1-introduction/examples/.venv/lib/python3.13/site-packages/stable_baselines3/common/evaluation.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


CartPole settings on MountainCar: -200.0 +/- 0.0   (expected: stuck at -200)


## Exercise 1 continued - the rl-zoo TUNED MountainCar hyperparameters

Expected result: roughly **-100 to -130** (reaches the flag in 100-130 steps). Exact numbers vary by machine - same seed, different floating-point arithmetic; RL amplifies the difference. Takes ~5 minutes on a CPU.

Note what changed: much more exploration (fraction 0.2, final eps 0.07), a small buffer that stays fresh (10k), frequent target updates relative to the update rhythm - and gamma 0.98. These are the values in RL Lab's `backend/algorithms/dqn.py`.

In [2]:
tuned_settings = dict(
    learning_rate=4e-3, gamma=0.98, batch_size=128,
    exploration_fraction=0.2, exploration_final_eps=0.07,
    buffer_size=10_000, learning_starts=1000, target_update_interval=600,
    train_freq=16, gradient_steps=8, policy_kwargs={'net_arch': [256, 256]},
)

model = DQN('MlpPolicy', env, seed=42, verbose=0, **tuned_settings)
model.learn(total_timesteps=100_000, progress_bar=True)
mean, std = evaluate_policy(model, env, n_eval_episodes=20, deterministic=True)
print(f"Tuned settings on MountainCar:    {mean:.1f} +/- {std:.1f}   (expected: around -100 to -130)")

Tuned settings on MountainCar:    -166.9 +/- 48.3   (expected: around -100 to -130)


*Why one lucky success changes everything for DQN:* the replay buffer stores it, and the network trains on it thousands of times afterwards - a single rare event becomes a permanent teacher. PPO (Exercise 2) throws its rollouts away after each update, so a lucky success helps once and is forgotten.

## Exercise 2 - PPO on CartPole, then on MountainCar

CartPole expected result: around **500** (SB3's PPO defaults are solid on CartPole).
MountainCar expected result: stuck at **-200** - and unlike DQN, more steps rarely help: with every rollout returning exactly -200, all advantage estimates are zero and the policy gradient carries no information at all.

In [3]:
from stable_baselines3 import PPO

# --- CartPole: works with plain defaults ---
env_cp = gym.make('CartPole-v1', render_mode='rgb_array')
model = PPO('MlpPolicy', env_cp, seed=42, verbose=0)
model.learn(total_timesteps=50_000, progress_bar=True)
mean, std = evaluate_policy(model, env_cp, n_eval_episodes=20, deterministic=True)
print(f"PPO on CartPole:    {mean:.1f} +/- {std:.1f}   (expected: ~500)")

# --- MountainCar: expected to fail (see RL Lab's environment page for why) ---
env_mc = gym.make('MountainCar-v0', render_mode='rgb_array')
model = PPO('MlpPolicy', env_mc, seed=42, verbose=0)
model.learn(total_timesteps=50_000, progress_bar=True)
mean, std = evaluate_policy(model, env_mc, n_eval_episodes=20, deterministic=True)
print(f"PPO on MountainCar: {mean:.1f} +/- {std:.1f}   (expected: stuck at -200)")

PPO on CartPole:    500.0 +/- 0.0   (expected: ~500)


PPO on MountainCar: -200.0 +/- 0.0   (expected: stuck at -200)
